# Case 01 — The M3 Subscription Cliff
**Industry:** E-commerce (Subscription SaaS) | **Level:** Intermediate

---

## 1. Business Background

**GlowBox** is a premium monthly beauty subscription delivering curated skincare and makeup to **120,000 active subscribers** across the US.

- **New signups:** ~10,000/month
- **CAC (Customer Acquisition Cost):** $60 per subscriber
- **Monthly margin per subscriber:** $15
- **Break-even point:** Month 4 (needs 4 × $15 = $60 to recover CAC)
- **Problem:** 45% of new subscribers churn by Month 3 — before ever becoming profitable

**Industry benchmark** for M3 churn in D2C beauty boxes: ~38%. GlowBox at 45% is 7pp above market.

**Current burn:** $600K/month acquiring customers who leave before ROI.

---

## 2. Business Problem — The Three Hypotheses

| Hypothesis | Description |
|---|---|
| H1 | "Discount hunters" — signed up for a 50%-off promo, never intended to stay |
| H2 | Value/excitement drops after the "honeymoon phase" of boxes 1–2 |
| H3 | Low engagement in Month 1 → customers forget they're subscribed → shocked by M3 charge |

**Direct financial loss:** 4,500 churned users × $15 = **$67,500/month** in lost margin  
**Wasted CAC:** 4,500 × $60 = **$270,000/month** that never generated ROI

---

## 3. Analytics Objective

Move from **reactive save-the-customer** calls → **proactive churn prevention**.

Key questions:
1. Which Month-1 behaviors predict Month-3 churn? (Leading vs. lagging indicators)
2. Does the first box product mix affect long-term retention?
3. Can we separate "discount hunters" (unrecoverable) from "disengaged subscribers" (saveable)?

**Target:** Reduce M3 churn from 45% → 35% (saves ~$112,500/month in direct margin)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
sns.set_theme(style='whitegrid', palette='muted')
print('Libraries loaded.')

---
## 4. Data Simulation

We simulate a cohort of **10,000 subscribers** with the key variables from the case:

| Variable | Type | Role |
|---|---|---|
| `signup_source` | Categorical (TikTok/IG/Search/Referral) | Channel quality signal |
| `first_box_category` | Binary (Skincare/Makeup) | Product-fit signal |
| `discount_used` | Binary | Intent proxy (H1) |
| `login_frequency` | Continuous (days active M1) | Engagement leading indicator |
| `choice_made` | Binary | Investment/ownership signal |
| `email_open_rate` | Continuous [0,1] | Engagement leading indicator |
| `review_score` | Ordinal 1–5 | Lagging satisfaction indicator |
| `signup_month` | Categorical (Holiday/Spring) | Cohort signal |

In [ ]:
N = 10_000

# Signup source (affects churn probability)
sources = np.random.choice(['TikTok', 'Instagram', 'Search', 'Referral'],
                           size=N, p=[0.35, 0.30, 0.20, 0.15])

# Signup month — Holiday cohort (Nov/Dec) vs Spring (Apr/May) vs Other
signup_month = np.random.choice(['Holiday', 'Spring', 'Other'],
                                 size=N, p=[0.22, 0.18, 0.60])

# Discount used — holiday cohort and TikTok users are more likely
discount_prob = np.where(signup_month == 'Holiday', 0.78,
                np.where(sources == 'TikTok', 0.60,
                np.where(sources == 'Referral', 0.15, 0.35)))
discount_used = np.random.binomial(1, discount_prob)

# First box category — slightly more makeup
box_category = np.random.choice(['Skincare', 'Makeup'], size=N, p=[0.48, 0.52])

# Login frequency M1 — skincare users and referral users engage more
base_login = np.where(box_category == 'Skincare', 8, 5)
base_login = base_login + np.where(sources == 'Referral', 3, 0)
base_login = base_login - np.where(discount_used == 1, 2, 0)
login_frequency = np.random.poisson(base_login).clip(0, 30)

# Choice made — driven by engagement (login)
choice_prob = (0.05 + 0.06 * login_frequency).clip(0, 0.95)
choice_made = np.random.binomial(1, choice_prob)

# Email open rate
base_email = 0.30 + 0.025 * login_frequency - 0.10 * discount_used
email_open_rate = np.random.beta(2, 5, size=N).clip(0, 1) * 0.5 + base_email.clip(0, 0.8) * 0.5
email_open_rate = email_open_rate.clip(0, 1)

# Review score — mostly satisfaction noise (lagging indicator)
review_score = np.random.choice([1, 2, 3, 4, 5], size=N, p=[0.05, 0.10, 0.20, 0.40, 0.25])

# ------------------------------------------------------------------
# TARGET: churn_m3 — calibrated to match case statistics
# Key known rates: promo → 58%, no-promo → 31%
#                 skincare → 30%, makeup → 55%
#                 login <5 → 62%, login ≥5 → 22%
#                 choice yes → 18%, choice no → 57%
# ------------------------------------------------------------------
log_odds = (
    0.30                                         # baseline intercept
    + 0.80 * discount_used                       # promo signal
    + 0.90 * (box_category == 'Makeup')          # makeup churn
    - 0.12 * login_frequency                     # engagement lowers churn
    - 1.50 * choice_made                         # choice = strong retention
    - 0.60 * email_open_rate                     # email engagement helps
    + 0.40 * (sources == 'TikTok')              # TikTok = lower quality
    - 0.45 * (sources == 'Referral')            # referral = higher quality
    + 0.30 * (signup_month == 'Holiday')         # holiday cohort worse
)
churn_prob = 1 / (1 + np.exp(-log_odds))
churn_m3 = np.random.binomial(1, churn_prob)

df = pd.DataFrame({
    'signup_source': sources,
    'signup_month': signup_month,
    'first_box_category': box_category,
    'discount_used': discount_used,
    'login_frequency': login_frequency,
    'choice_made': choice_made,
    'email_open_rate': email_open_rate.round(3),
    'review_score': review_score,
    'churn_m3': churn_m3
})

print(f'Dataset: {len(df):,} subscribers')
print(f'Overall M3 churn rate: {df.churn_m3.mean():.1%}')
print(f'\nSample (first 10 rows):')
df.head(10)

---
## 5. Step 1 — Customer Lifecycle Funnel

**Before any modeling:** map WHERE the biggest drop-off occurs.

From the case data:
- **M1 drop (signup → M1 active):** 8% (silent disengagement early)
- **M2 drop (M1 → M2 active):** 15%
- **M3 CLIFF (M2 → M3 retained):** **30%** ← this is the problem
- **M4 drop (M3 → profitable):** 7%

In [ ]:
stages = ['Signup\n(10,000)', 'M1 Active\n(9,200)', 'M2 Active\n(7,800)',
          'M3 Retained\n(5,500)', 'M4+ Profitable\n(5,100)']
counts = [10000, 9200, 7800, 5500, 5100]
drop_labels = ['', '-8%\n(800)', '-15%\n(1,400)', '-30% ← THE CLIFF\n(2,300)', '-7%\n(400)']

fig, ax = plt.subplots(figsize=(12, 5))
colors = ['#4C72B0', '#55A868', '#C44E52', '#DD8452', '#8172B2']

bars = ax.bar(stages, counts, color=colors, width=0.55, edgecolor='white', linewidth=1.5)

for bar, count, drop in zip(bars, counts, drop_labels):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 100,
            f'{count:,}', ha='center', va='bottom', fontsize=11, fontweight='bold')
    if drop:
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() / 2,
                drop, ha='center', va='center', fontsize=9,
                color='white', fontweight='bold')

ax.set_title('GlowBox Customer Lifecycle Funnel\nWhere does the drop-off concentrate?',
             fontsize=13, fontweight='bold', pad=15)
ax.set_ylabel('Number of Subscribers', fontsize=11)
ax.set_ylim(0, 12000)
ax.spines[['top', 'right']].set_visible(False)

# Highlight the M3 cliff
ax.axvline(x=2.5, color='red', linestyle='--', linewidth=2, alpha=0.7)
ax.text(2.55, 11500, 'M3 CLIFF\n(Primary Focus)', color='red', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

print('Key insight: The M2 → M3 transition has the largest absolute drop (2,300 users = 30%).')
print('BUT: M1 → M2 also drops 1,400 — a secondary "silent drop" to investigate.')

---
## 6. Step 2 — Cohort Analysis

Group subscribers by **signup month**. Key hypothesis: Holiday (Nov/Dec) cohort attracts deal-seekers, not brand loyalists.

In [ ]:
cohort_stats = df.groupby('signup_month').agg(
    n_subscribers=('churn_m3', 'count'),
    m3_churn_rate=('churn_m3', 'mean'),
    discount_rate=('discount_used', 'mean'),
    avg_logins=('login_frequency', 'mean'),
    choice_rate=('choice_made', 'mean')
).round(3).reset_index()

print('=== Cohort Analysis ===')
print(cohort_stats.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

cohort_order = ['Holiday', 'Spring', 'Other']
cohort_df = cohort_stats.set_index('signup_month').loc[cohort_order]

axes[0].bar(cohort_order, cohort_df['m3_churn_rate'] * 100,
            color=['#C44E52', '#55A868', '#4C72B0'], edgecolor='white')
axes[0].axhline(38, color='gray', linestyle='--', label='Industry benchmark (38%)')
axes[0].set_title('M3 Churn Rate by Signup Cohort', fontweight='bold')
axes[0].set_ylabel('M3 Churn Rate (%)')
axes[0].legend()
for i, (cohort, row) in enumerate(cohort_df.iterrows()):
    axes[0].text(i, row['m3_churn_rate'] * 100 + 0.5, f"{row['m3_churn_rate']:.1%}",
                 ha='center', fontweight='bold')

axes[1].bar(cohort_order, cohort_df['discount_rate'] * 100,
            color=['#C44E52', '#55A868', '#4C72B0'], edgecolor='white')
axes[1].set_title('% Using Promo Code by Cohort', fontweight='bold')
axes[1].set_ylabel('Promo Usage Rate (%)')
for i, (cohort, row) in enumerate(cohort_df.iterrows()):
    axes[1].text(i, row['discount_rate'] * 100 + 0.5, f"{row['discount_rate']:.1%}",
                 ha='center', fontweight='bold')

plt.suptitle('Holiday Cohort: More Promo Users → Higher Churn', fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## 7. Step 3 — Hypothesis Testing via Pivot Tables

Before building any model, test each hypothesis with simple segment comparisons.

> **Rule:** If the churn gap between two segments is large (>10pp), the variable is a strong signal worth including in the model.

In [ ]:
def churn_by_segment(df, col, label_yes, label_no, condition):
    yes = df[condition]['churn_m3'].mean()
    no = df[~condition]['churn_m3'].mean()
    n_yes = condition.sum()
    n_no = (~condition).sum()
    gap = yes - no
    return {
        'Segment (Yes)': label_yes, 'Churn (Yes)': f'{yes:.1%}', 'N (Yes)': n_yes,
        'Segment (No)': label_no,  'Churn (No)': f'{no:.1%}',  'N (No)': n_no,
        'Gap': f'{gap:+.1%}', 'Signal': '✓ STRONG' if abs(gap) > 0.15 else
                              '✓ MEDIUM' if abs(gap) > 0.07 else '✗ WEAK'
    }

segments = [
    churn_by_segment(df, 'discount_used', 'Promo Signup (Yes)', 'Promo Signup (No)',
                     df['discount_used'] == 1),
    churn_by_segment(df, 'first_box_category', 'Box: Skincare-heavy', 'Box: Makeup-heavy',
                     df['first_box_category'] == 'Skincare'),
    churn_by_segment(df, 'login_frequency', 'Login M1 ≥ 5 days', 'Login M1 < 5 days',
                     df['login_frequency'] >= 5),
    churn_by_segment(df, 'choice_made', 'Choice Made (Yes)', 'Choice Made (No)',
                     df['choice_made'] == 1),
    churn_by_segment(df, 'review_score', 'Review Score ≥ 4', 'Review Score < 4',
                     df['review_score'] >= 4),
]

pivot = pd.DataFrame(segments)
print('=== Hypothesis Testing: Churn Rate by Segment ===')
print(pivot[['Segment (Yes)', 'Churn (Yes)', 'N (Yes)', 'Churn (No)', 'N (No)', 'Gap', 'Signal']].to_string(index=False))
print()
print('Key surprise: review_score is the WEAKEST predictor.')
print('"Participation" (choice_made) beats "Satisfaction" (review_score) every time.')

In [ ]:
# Visualize churn rates by channel
channel_churn = df.groupby('signup_source')['churn_m3'].agg(['mean', 'count']).reset_index()
channel_churn = channel_churn.sort_values('mean', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Churn by channel
bar_colors = ['#C44E52' if c in ['TikTok', 'Instagram'] else '#55A868'
              for c in channel_churn['signup_source']]
axes[0].bar(channel_churn['signup_source'], channel_churn['mean'] * 100,
            color=bar_colors, edgecolor='white')
axes[0].set_title('M3 Churn Rate by Acquisition Channel', fontweight='bold')
axes[0].set_ylabel('M3 Churn Rate (%)')
axes[0].axhline(38, color='gray', linestyle='--', alpha=0.7, label='Industry benchmark')
axes[0].legend()
for i, row in channel_churn.iterrows():
    axes[0].text(list(channel_churn['signup_source']).index(row['signup_source']),
                 row['mean'] * 100 + 0.5, f"{row['mean']:.1%}", ha='center', fontweight='bold')

# Churn by box category
box_churn = df.groupby('first_box_category')['churn_m3'].mean()
axes[1].bar(box_churn.index, box_churn.values * 100,
            color=['#4C72B0', '#C44E52'], edgecolor='white')
axes[1].set_title('M3 Churn Rate by First Box Category', fontweight='bold')
axes[1].set_ylabel('M3 Churn Rate (%)')
for i, (cat, val) in enumerate(box_churn.items()):
    axes[1].text(i, val * 100 + 0.5, f'{val:.1%}', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

print('TikTok churns ~2× more than Referral: a channel quality problem, not a product problem.')

---
## 8. Step 4 — The "Silent Canceller" Discovery

**Finding:** Users who don't log in for 14+ days after Box 1 have an **85% probability of churning** — but they never complain.

Key insight: Compare **churned vs. retained** users at the same lifecycle stage (not just churned users in isolation — that's survivorship bias).

In [ ]:
# Login frequency distribution: churned vs retained
churned = df[df['churn_m3'] == 1]['login_frequency']
retained = df[df['churn_m3'] == 0]['login_frequency']

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Distribution comparison
axes[0].hist(churned, bins=25, alpha=0.65, color='#C44E52', label=f'Churned (n={len(churned):,})', density=True)
axes[0].hist(retained, bins=25, alpha=0.65, color='#55A868', label=f'Retained (n={len(retained):,})', density=True)
axes[0].axvline(5, color='black', linestyle='--', linewidth=1.5, label='Threshold: 5 logins')
axes[0].set_title('M1 Login Frequency: Churned vs. Retained', fontweight='bold')
axes[0].set_xlabel('Login Days in Month 1')
axes[0].set_ylabel('Density')
axes[0].legend()

# Churn rate by login bucket
df['login_bucket'] = pd.cut(df['login_frequency'], bins=[-1, 0, 2, 4, 7, 10, 30],
                             labels=['0', '1-2', '3-4', '5-7', '8-10', '11+'])
login_churn = df.groupby('login_bucket', observed=True)['churn_m3'].mean() * 100
colors_bar = ['#C44E52' if v > 50 else '#FFA500' if v > 30 else '#55A868'
              for v in login_churn.values]
axes[1].bar(login_churn.index.astype(str), login_churn.values, color=colors_bar, edgecolor='white')
axes[1].set_title('M3 Churn Rate by M1 Login Frequency', fontweight='bold')
axes[1].set_xlabel('Login Days in Month 1')
axes[1].set_ylabel('M3 Churn Rate (%)')
axes[1].axhline(38, color='gray', linestyle='--', alpha=0.7, label='Industry benchmark')
axes[1].legend()
for i, v in enumerate(login_churn.values):
    axes[1].text(i, v + 0.5, f'{v:.0f}%', ha='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

# Stats: churned vs retained login means
print(f'Churned users avg M1 logins:  {churned.mean():.1f} days')
print(f'Retained users avg M1 logins: {retained.mean():.1f} days')
t_stat, p_val = stats.ttest_ind(churned, retained)
print(f'T-test p-value: {p_val:.2e} → highly significant')
print()
print('"Silent Canceller" threshold: 0 logins in M1 → ~85% churn probability.')
print('These users never complain — they simply forget the subscription exists.')

---
## 9. Step 5 — Logistic Regression: Building the Churn Risk Score

We run a logistic regression with all **leading indicators** as inputs to build a churn probability score.

**Key diagnostic:** `login_frequency` coefficient should be negative (more logins → less churn). We also check for multicollinearity using VIF.

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
import statsmodels.api as sm

# Encode categoricals
df_model = df.copy()
df_model['is_tiktok'] = (df_model['signup_source'] == 'TikTok').astype(int)
df_model['is_referral'] = (df_model['signup_source'] == 'Referral').astype(int)
df_model['is_makeup'] = (df_model['first_box_category'] == 'Makeup').astype(int)
df_model['is_holiday'] = (df_model['signup_month'] == 'Holiday').astype(int)

feature_cols = ['discount_used', 'login_frequency', 'choice_made',
                'email_open_rate', 'review_score', 'is_tiktok', 'is_referral',
                'is_makeup', 'is_holiday']

X = df_model[feature_cols]
y = df_model['churn_m3']

# Logistic regression via statsmodels for interpretable coefficients
X_const = sm.add_constant(X)
logit_model = sm.Logit(y, X_const).fit(disp=False)

print('=== Logistic Regression Results ===')
results_df = pd.DataFrame({
    'Feature': feature_cols,
    'Coefficient': logit_model.params[1:].values.round(3),
    'P-value': logit_model.pvalues[1:].values.round(4),
    'Odds Ratio': np.exp(logit_model.params[1:].values).round(3)
})
results_df['Significant'] = results_df['P-value'].apply(
    lambda p: '*** p<0.001' if p < 0.001 else '** p<0.01' if p < 0.01 else
              '* p<0.05' if p < 0.05 else 'ns'
)
print(results_df.sort_values('Coefficient').to_string(index=False))
print()

# VIF check
vif_data = pd.DataFrame()
vif_data['Feature'] = feature_cols
vif_data['VIF'] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
print('=== VIF (Multicollinearity Check) ===')
high_vif = vif_data[vif_data['VIF'] > 5]
if len(high_vif) == 0:
    print('All VIF < 5: No multicollinearity concern.')
else:
    print(f'{len(high_vif)} feature(s) with VIF > 5 — multicollinearity present; interpret those coefficients with caution:')
    print(high_vif.sort_values('VIF', ascending=False).to_string(index=False))
    print()
print('Full VIF table:')
print(vif_data.sort_values('VIF', ascending=False).to_string(index=False))

In [ ]:
# Visualize coefficients
fig, ax = plt.subplots(figsize=(9, 5))
results_sorted = results_df.sort_values('Coefficient')
colors = ['#C44E52' if c > 0 else '#55A868' for c in results_sorted['Coefficient']]

bars = ax.barh(results_sorted['Feature'], results_sorted['Coefficient'],
               color=colors, edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Logistic Regression Coefficients\n(Positive = Increases Churn Risk)', fontweight='bold')
ax.set_xlabel('Log-Odds Coefficient')

for bar, row in zip(bars, results_sorted.itertuples()):
    if row.Significant != 'ns':
        xpos = row.Coefficient + (0.02 if row.Coefficient > 0 else -0.02)
        ax.text(xpos, bar.get_y() + bar.get_height() / 2,
                row.Significant.split(' ')[0], va='center', fontsize=9, color='darkred')

red_patch = mpatches.Patch(color='#C44E52', label='Increases churn risk')
green_patch = mpatches.Patch(color='#55A868', label='Decreases churn risk')
ax.legend(handles=[red_patch, green_patch])
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()

---
## 10. Step 6 — Financial Impact Calculation

Translate churn reduction into dollar impact across **three scenarios** (Conservative / Target / Optimistic).

**Key assumptions:**
- CAC = $60 | Monthly margin = $15 | Break-even = Month 4
- Current M3 churn = 45% (4,500 churned out of 10,000/month)
- Intervention cost = $27,000/month (email platform + COGS increase)

In [ ]:
CAC = 60
MONTHLY_MARGIN = 15
NEW_SUBS_PER_MONTH = 10_000
CURRENT_CHURN = 0.45
CURRENT_CHURNED = int(NEW_SUBS_PER_MONTH * CURRENT_CHURN)  # 4,500
INTERVENTION_COST_MONTHLY = 27_000  # email + COGS delta

scenarios = [
    ('Conservative', 0.40),
    ('Target',       0.35),
    ('Optimistic',   0.30),
]

rows = []
for name, new_churn in scenarios:
    users_saved = int((CURRENT_CHURN - new_churn) * NEW_SUBS_PER_MONTH)
    margin_saved = users_saved * MONTHLY_MARGIN
    revenue_m4_m6 = users_saved * MONTHLY_MARGIN * 3  # 3 extra months kept
    gross_benefit = margin_saved + revenue_m4_m6
    net_benefit = gross_benefit - INTERVENTION_COST_MONTHLY
    rows.append({
        'Scenario': name,
        'New Churn': f'{new_churn:.0%}',
        'Users Saved': f'{users_saved:,}',
        'Margin Saved/mo': f'${margin_saved:,}',
        'Revenue M4-M6': f'${revenue_m4_m6:,}',
        'Gross Benefit': f'${gross_benefit:,}',
        'Intervention Cost': f'${INTERVENTION_COST_MONTHLY:,}',
        'Net Benefit/mo': f'${net_benefit:,}',
        'Annual Value': f'${net_benefit * 12:,}'
    })

results_table = pd.DataFrame(rows)
print('=== Sensitivity Analysis: Financial Impact of Churn Reduction ===')
print(results_table.to_string(index=False))

print()
print('Current state:')
print(f'  Lost margin/month:  ${CURRENT_CHURNED * MONTHLY_MARGIN:,}')
print(f'  Wasted CAC/month:   ${CURRENT_CHURNED * CAC:,} (never recovered)')

In [ ]:
# LTV / CAC trajectory
print('=== LTV / CAC Analysis ===')

def calc_ltv_cac(avg_retention_months, monthly_margin=15, cac=60):
    ltv = monthly_margin * avg_retention_months
    return ltv, ltv / cac

# Current: 45% churn → avg ~3.7 months retention
ltv_current, ratio_current = calc_ltv_cac(3.7)
# Target (35% churn) → avg ~5.4 months
ltv_target, ratio_target = calc_ltv_cac(5.4)
# Series B requirement: LTV/CAC ≥ 3.0 → avg 12 months
ltv_seriesb, ratio_seriesb = calc_ltv_cac(12)

print(f'Current state (45% M3 churn): LTV = ${ltv_current:.0f} | LTV/CAC = {ratio_current:.2f}')
print(f'Target (35% M3 churn):        LTV = ${ltv_target:.0f} | LTV/CAC = {ratio_target:.2f}')
print(f'Series B requirement:          LTV = ${ltv_seriesb:.0f} | LTV/CAC = {ratio_seriesb:.2f} (need avg 12-month retention)')
print()
print('Honest conclusion: This program moves LTV/CAC from 0.92 → ~1.35.')
print('Getting to 3.0 requires a multi-year retention roadmap, not a single intervention.')

# Waterfall chart
fig, ax = plt.subplots(figsize=(8, 5))
states = ['Current\n(0.92)', 'After Program\n(~1.35)', 'Series B Target\n(3.0)']
values = [ratio_current, ratio_target, ratio_seriesb]
colors = ['#C44E52', '#FFA500', '#55A868']
bars = ax.bar(states, values, color=colors, edgecolor='white', width=0.5)
ax.axhline(1.0, color='gray', linestyle='--', linewidth=1.2, label='Break-even (LTV/CAC = 1.0)')
ax.axhline(3.0, color='green', linestyle='--', linewidth=1.2, label='Series B target (3.0)')
ax.set_title('LTV/CAC Ratio: Current vs. Program Impact vs. Series B Target', fontweight='bold')
ax.set_ylabel('LTV / CAC Ratio')
for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width() / 2, val + 0.05, f'{val:.2f}', ha='center', fontweight='bold')
ax.legend()
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()

---
## 11. A/B Test Design — Resolving Causality

Observation ≠ Causation. We saw skincare subscribers churn less — but maybe skincare-interested users simply have higher baseline loyalty, regardless of box contents.

**Test design:**
- **Treatment:** Skincare-heavy M1 box + choice-nudge email (sent 3 days after delivery)
- **Control:** Standard M1 box (no nudge)
- **Sample:** 2,000 new signups, 50/50 random split
- **Primary metric:** M3 retention rate
- **Decision rule:** If treatment M3 retention > control by ≥5pp (p < 0.05) → full rollout

In [ ]:
# Simulate the A/B test result
np.random.seed(99)
n_per_group = 1_000

# Control: standard box, no nudge → ~45% churn (55% retention)
control_retained = np.random.binomial(n_per_group, 0.55)
# Treatment: skincare box + nudge → ~35% churn (65% retention)
treatment_retained = np.random.binomial(n_per_group, 0.65)

control_rate = control_retained / n_per_group
treatment_rate = treatment_retained / n_per_group
lift = treatment_rate - control_rate

# Two-proportion z-test
p_pool = (control_retained + treatment_retained) / (2 * n_per_group)
se = np.sqrt(p_pool * (1 - p_pool) * (1/n_per_group + 1/n_per_group))
z_score = lift / se
p_value = 2 * (1 - stats.norm.cdf(abs(z_score)))

print('=== A/B Test Results (Simulated) ===')
print(f'Control   — Retained: {control_retained:,}/{n_per_group:,} = {control_rate:.1%}')
print(f'Treatment — Retained: {treatment_retained:,}/{n_per_group:,} = {treatment_rate:.1%}')
print(f'Lift: {lift:+.1%} | Z-score: {z_score:.2f} | p-value: {p_value:.4f}')
print()
if p_value < 0.05 and lift >= 0.05:
    print('✓ Decision: FULL ROLLOUT — Lift ≥ 5pp and statistically significant (p < 0.05)')
elif p_value < 0.05:
    print('→ Decision: Effect is significant but <5pp. Run follow-up 2×2 factorial test.')
else:
    print('✗ Decision: No significant effect. Pivot to Churn Risk Dashboard as primary intervention.')

---
## 12. Key Findings Summary

| Finding | Expected | Actual |
|---|---|---|
| Strongest retention predictor | Review score (satisfaction) | `login_frequency` + `choice_made` (behavior) |
| Churn uniform across channels | Equal rates | TikTok churns 2× vs. Referral — a channel quality issue |
| Product type doesn't matter | Similar retention | Skincare 30% vs. Makeup 55% — 40% LTV differential |
| High ratings prevent churn | Yes | No — users love the product but still cancel |

## 13. Recommendations Tier Summary

| Tier | Action | Expected Impact | Cost |
|---|---|---|---|
| **P0 (0–30 days)** | Trigger "Personalize your next box" email 3 days post-delivery | +Choice Made: 31% → 50% | $3K/mo |
| **P0** | Silent Canceller alert (0 logins in 14 days) → push notification | Recover 15% of silent cancellers | Minimal |
| **P1 (1–6 months)** | Shift M1 box to skincare-heavy (after A/B test) | Convert 20% of makeup churners | +$2.40/box COGS |
| **P1** | Reallocate 15% TikTok budget → Referral incentives | Improve new cohort avg LTV | $27K/mo redirected |
| **P2 (6+ months)** | Build in-app "Skin Diary" feature | +3–4 months avg retention for skincare users | $80–120K dev |

---

## 14. Senior Practitioner's Lesson

> **The Reframe:** Management asked *"How do we reduce churn?"*  
> A great analyst asks instead: *"How do we embed this product into a daily routine before the novelty threshold at M3?"*
> 
> The first framing leads to save campaigns and discounts.  
> The second leads to onboarding redesign, habit architecture, and psychological ownership — which is what `choice_made` represents.

**Interview framework:** Funnel → Cohort → Segment → Signal → Experiment